# TANGO Training Notebook (84M UNet, AudioCaps, T4)

**What this notebook does:**
1. Mounts Google Drive (checkpoints + data survive session resets)
2. Clones / pulls the latest code from the repo
3. Installs dependencies
4. Downloads 1000 AudioCaps training clips from YouTube
5. Encodes audio → VAE latents (saved to Drive)
6. Trains the 84M UNet for 20 epochs, logging to TensorBoard
7. Shows live loss curves

> **Runtime:** Make sure you have a **T4 GPU** runtime selected  
> Runtime → Change runtime type → T4 GPU

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT   = '/content/drive/MyDrive/tango'
DATA_DIR     = f'{DRIVE_ROOT}/train_latents'    # encoded latents live here
OUTPUT_DIR   = f'{DRIVE_ROOT}/runs/small_unet'  # checkpoints + tensorboard
AUDIOLDM_CKPT = f'{DRIVE_ROOT}/audioldm-s-full.ckpt'  # cached VAE checkpoint

for d in [DRIVE_ROOT, DATA_DIR, OUTPUT_DIR, os.path.dirname(AUDIOLDM_CKPT)]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')
print(f'  data    → {DATA_DIR}')
print(f'  runs    → {OUTPUT_DIR}')
print(f'  vae     → {AUDIOLDM_CKPT}')

## 2 — Clone / pull repo

In [ ]:
%%bash
BRANCH="claude/fix-wavcaps-file-matching-GCdR0"
REPO="https://github.com/galos123/Tango.git"

if [ -d /content/Tango/.git ]; then
    echo "Repo already cloned — pulling latest..."
    cd /content/Tango
    git fetch origin $BRANCH
    git checkout $BRANCH
    git pull origin $BRANCH
else
    echo "Cloning repo..."
    git clone --branch $BRANCH $REPO /content/Tango
fi

echo "Done. Current commit:"
cd /content/Tango && git log --oneline -3

## 3 — Install dependencies

In [ ]:
%%bash
pip install -q \
    diffusers==0.21.0 \
    transformers \
    accelerate \
    soundfile \
    librosa \
    tqdm \
    huggingface_hub \
    scipy \
    pandas \
    yt-dlp

echo "All packages installed."

## 4 — Download 1000 AudioCaps training clips

This step downloads audio from YouTube.  
**Expected time: 30–60 minutes** depending on connection speed.  
Already-downloaded files are skipped on re-run (resumable).

In [ ]:
import os, csv, subprocess, re
from pathlib import Path
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
N_TRAIN_SAMPLES = 1000
AUDIO_DIR = '/content/audiocaps_train_audio'
TRAIN_CSV = '/content/audiocaps_train.csv'
os.makedirs(AUDIO_DIR, exist_ok=True)

# ── Download train CSV ────────────────────────────────────────────────────────
if not os.path.exists(TRAIN_CSV):
    subprocess.run([
        'wget', '-q', '-O', TRAIN_CSV,
        'https://raw.githubusercontent.com/cdjkim/audiocaps/master/dataset/train.csv'
    ], check=True)
    print(f'Downloaded train CSV.')

# ── Read first N_TRAIN_SAMPLES entries ────────────────────────────────────────
rows = []
with open(TRAIN_CSV, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)
        if len(rows) >= N_TRAIN_SAMPLES:
            break
print(f'Loaded {len(rows)} entries from train CSV.')

# ── Download audio with yt-dlp ─────────────────────────────────────────────
ok, skipped, failed = 0, 0, 0
for row in tqdm(rows, desc='Downloading'):
    ytid    = row['youtube_id']
    start   = float(row['start_time'])
    end     = float(row['end_time'])
    out_path = os.path.join(AUDIO_DIR, f'{ytid}.wav')

    if os.path.exists(out_path):
        skipped += 1
        continue

    url = f'https://www.youtube.com/watch?v={ytid}'
    tmp = os.path.join(AUDIO_DIR, f'_tmp_{ytid}')

    ret = subprocess.run([
        'yt-dlp', '-q', '--no-warnings',
        '-x', '--audio-format', 'wav',
        '--audio-quality', '0',
        '--postprocessor-args', f'-ar 16000 -ac 1 -ss {start} -to {end}',
        '-o', tmp + '.%(ext)s',
        url
    ], capture_output=True, timeout=60)

    # find the output file (yt-dlp adds extension)
    candidates = list(Path(AUDIO_DIR).glob(f'_tmp_{ytid}.*'))
    if ret.returncode == 0 and candidates:
        candidates[0].rename(out_path)
        ok += 1
    else:
        for c in candidates:
            c.unlink(missing_ok=True)
        failed += 1

total = ok + skipped
print(f'\nDownloaded: {ok}  |  Skipped (already exist): {skipped}  |  Failed: {failed}')
print(f'Total available: {total} audio files')
if total < 100:
    print('WARNING: fewer than 100 files — some YouTube videos may be unavailable.')

## 5 — Encode audio → VAE latents

Runs the AudioLDM VAE encoder on every downloaded clip.  
Output saved to **Google Drive** so this only needs to run once.  
**Expected time: ~15–20 minutes** for 1000 files on T4.

In [ ]:
import sys, importlib.util, csv, os
import torch
from pathlib import Path
from tqdm import tqdm

sys.path.insert(0, '/content/Tango')

# ── Import VAE + CONFIG + STFT from the latent-creation script ────────────────
_spec = importlib.util.spec_from_file_location(
    'latent_creation',
    '/content/Tango/a-latent-creation-wavcaps.py'
)
_lc = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_lc)

AutoencoderKL = _lc.AutoencoderKL
TacotronSTFT  = _lc.TacotronSTFT
wav_to_fbank  = _lc.wav_to_fbank
CONFIG        = _lc.CONFIG
download_file = _lc.download_file
AUDIO_LDM_URL = _lc.AUDIO_LDM_S_FULL_URL

# ── Download / reuse cached VAE checkpoint ────────────────────────────────────
if not os.path.exists(AUDIOLDM_CKPT) or os.path.getsize(AUDIOLDM_CKPT) < 100_000_000:
    print(f'Downloading AudioLDM checkpoint to {AUDIOLDM_CKPT} (~2.5 GB)...')
    download_file(AUDIO_LDM_URL, AUDIOLDM_CKPT, min_bytes_ok=100_000_000)
else:
    print(f'Using cached checkpoint: {AUDIOLDM_CKPT}')

# ── Build VAE ─────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

ckpt = torch.load(AUDIOLDM_CKPT, map_location='cpu')
state = ckpt.get('state_dict', ckpt)

scale_factor = 1.0
for k in ['scale_factor', 'model.scale_factor']:
    if k in state:
        scale_factor = float(state[k].item() if torch.is_tensor(state[k]) else state[k])
        print(f'scale_factor = {scale_factor}')
        break

prefix = 'first_stage_model.'
vae_state = {k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)}
del ckpt, state

vae_cfg = CONFIG['first_stage_config']['params']
vae = AutoencoderKL(
    ddconfig=vae_cfg['ddconfig'],
    embed_dim=vae_cfg['embed_dim'],
    image_key=vae_cfg['image_key'],
    subband=vae_cfg['subband'],
    scale_factor=scale_factor,
)
vae.load_state_dict(vae_state, strict=False)
del vae_state
vae = vae.half().to(device).eval()
print('VAE ready.')

# ── Build STFT ────────────────────────────────────────────────────────────────
fn_STFT = TacotronSTFT(
    CONFIG['preprocessing']['stft']['filter_length'],
    CONFIG['preprocessing']['stft']['hop_length'],
    CONFIG['preprocessing']['stft']['win_length'],
    CONFIG['preprocessing']['mel']['n_mel_channels'],
    CONFIG['preprocessing']['audio']['sampling_rate'],
    CONFIG['preprocessing']['mel']['mel_fmin'],
    CONFIG['preprocessing']['mel']['mel_fmax'],
)
target_length = CONFIG['preprocessing']['mel']['target_length']

# ── Output dirs ───────────────────────────────────────────────────────────────
lat_dir = os.path.join(DATA_DIR, 'latent_vectors')
cap_dir = os.path.join(DATA_DIR, 'captions')
os.makedirs(lat_dir, exist_ok=True)
os.makedirs(cap_dir, exist_ok=True)

# ── Read CSV to build id→caption map ─────────────────────────────────────────
caption_map = {}   # youtube_id → caption
with open(TRAIN_CSV, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        caption_map[row['youtube_id']] = row['caption']
        if len(caption_map) >= N_TRAIN_SAMPLES:
            break

# ── Encode loop ───────────────────────────────────────────────────────────────
ok = skipped = errors = 0

audio_files = list(Path(AUDIO_DIR).glob('*.wav'))
print(f'Found {len(audio_files)} audio files to encode.')

for wav_path in tqdm(audio_files, desc='Encoding'):
    ytid = wav_path.stem
    lat_path = os.path.join(lat_dir, f'{ytid}.pt')

    if os.path.exists(lat_path):
        skipped += 1
        continue

    caption = caption_map.get(ytid)
    if caption is None:
        errors += 1
        continue

    try:
        mel, _, _ = wav_to_fbank(str(wav_path), target_length=target_length, fn_STFT=fn_STFT)
        x = mel.unsqueeze(0).unsqueeze(0).to(device, dtype=torch.float16)
        with torch.no_grad():
            posterior = vae.encode(x)
            z = vae.get_first_stage_encoding(posterior)  # (1, 8, H, W)
        torch.save(z.squeeze(0).cpu(), lat_path)
        with open(os.path.join(cap_dir, f'{ytid}.txt'), 'w') as f:
            f.write(caption)
        ok += 1
    except Exception as e:
        errors += 1
        if errors <= 5:
            print(f'  [WARN] {ytid}: {e}')

print(f'\nEncoded: {ok}  |  Skipped: {skipped}  |  Errors: {errors}')
print(f'Latents saved to: {lat_dir}')

del vae
torch.cuda.empty_cache()

## 6 — Train (84M UNet, 20 epochs)

**Expected time: ~80–100 minutes** on T4.  
Checkpoints are saved to Google Drive after every epoch.  
If Colab disconnects, re-run this cell — it will resume from `last.pt` automatically.

In [ ]:
import os, sys, json, re
from pathlib import Path

CKPT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
last_ckpt = os.path.join(CKPT_DIR, 'last.pt')
resume    = last_ckpt if os.path.exists(last_ckpt) else None

if resume:
    print(f'Resuming from: {resume}')
else:
    print('Starting fresh training run.')

# Write a temporary config override so train.py uses our settings
config = {
    'DATA_DIR':       DATA_DIR,
    'OUTPUT_DIR':     OUTPUT_DIR,
    'UNET_CONFIG':    '/content/Tango/unet_config.json',
    'PRETRAINED_HF':  None,
    'PRETRAINED_CKPT': None,
    'RESUME':         resume,
    'TEXT_ENCODER':   'google/flan-t5-large',
    'SNR_GAMMA':      5.0,
    'UNCONDITION':    False,
    'EPOCHS':         20,
    'BATCH_SIZE':     4,
    'LR':             1e-4,
    'WEIGHT_DECAY':   1e-2,
    'GRAD_CLIP':      1.0,
    'GRAD_ACCUM':     8,
    'WARMUP_STEPS':   100,
    'VAL_SPLIT':      0.1,
    'NUM_WORKERS':    2,
    'SAVE_EVERY':     5,
    'SEED':           42,
}

# Patch train.py __main__ block with our values and run as a module
sys.path.insert(0, '/content/Tango')
import importlib, types

# Read train.py source
src_path = '/content/Tango/train.py'
with open(src_path) as f:
    src = f.read()

# Replace the __main__ config block values
replacements = {
    r'DATA_DIR\s*=\s*[^\n]+':      f'DATA_DIR   = {repr(config["DATA_DIR"])}',
    r'OUTPUT_DIR\s*=\s*[^\n]+':    f'OUTPUT_DIR = {repr(config["OUTPUT_DIR"])}',
    r'UNET_CONFIG\s*=\s*[^\n]+':   f'UNET_CONFIG = {repr(config["UNET_CONFIG"])}',
    r'PRETRAINED_HF\s*=\s*[^\n]+': f'PRETRAINED_HF   = {repr(config["PRETRAINED_HF"])}',
    r'PRETRAINED_CKPT\s*=\s*[^\n]+':f'PRETRAINED_CKPT = {repr(config["PRETRAINED_CKPT"])}',
    r'RESUME\s*=\s*[^\n]+':        f'RESUME          = {repr(config["RESUME"])}',
    r'EPOCHS\s*=\s*[^\n]+':        f'EPOCHS       = {config["EPOCHS"]}',
    r'BATCH_SIZE\s*=\s*[^\n]+':    f'BATCH_SIZE   = {config["BATCH_SIZE"]}',
    r'LR\s*=\s*[^\n]+':            f'LR           = {config["LR"]}',
    r'WEIGHT_DECAY\s*=\s*[^\n]+':  f'WEIGHT_DECAY = {config["WEIGHT_DECAY"]}',
    r'GRAD_ACCUM\s*=\s*[^\n]+':    f'GRAD_ACCUM   = {config["GRAD_ACCUM"]}',
    r'WARMUP_STEPS\s*=\s*[^\n]+':  f'WARMUP_STEPS = {config["WARMUP_STEPS"]}',
    r'VAL_SPLIT\s*=\s*[^\n]+':     f'VAL_SPLIT    = {config["VAL_SPLIT"]}',
    r'SAVE_EVERY\s*=\s*[^\n]+':    f'SAVE_EVERY   = {config["SAVE_EVERY"]}',
}
patched = src
for pattern, repl in replacements.items():
    patched = re.sub(pattern, repl, patched)

patched_path = '/content/train_patched.py'
with open(patched_path, 'w') as f:
    f.write(patched)

print('Config applied. Starting training...\n')
exec(open(patched_path).read())

## 7 — Live TensorBoard

Run this cell **in parallel** with training (open it in a separate tab or run it before starting training).  
Refresh TensorBoard to see updated loss curves.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{OUTPUT_DIR}/tensorboard"

## 8 — Plot loss curves (no TensorBoard needed)

Alternative to TensorBoard — reads the event files and plots with matplotlib.

In [ ]:
import os
from pathlib import Path

try:
    from tensorflow.python.summary.summary_iterator import summary_iterator
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'tensorflow-cpu'], check=True)
    from tensorflow.python.summary.summary_iterator import summary_iterator

import matplotlib.pyplot as plt

tb_dir = Path(OUTPUT_DIR) / 'tensorboard'
event_files = sorted(tb_dir.glob('events.out.tfevents.*'))

if not event_files:
    print(f'No TensorBoard event files found in {tb_dir}.')
    print('Training may not have started yet.')
else:
    train_steps, train_losses = [], []
    val_steps,   val_losses   = [], []

    for ef in event_files:
        for event in summary_iterator(str(ef)):
            for v in event.summary.value:
                if v.tag == 'train/loss_epoch':
                    train_steps.append(event.step)
                    train_losses.append(v.simple_value)
                elif v.tag == 'val/loss_epoch':
                    val_steps.append(event.step)
                    val_losses.append(v.simple_value)

    fig, ax = plt.subplots(figsize=(10, 4))
    if train_losses:
        ax.plot(train_steps, train_losses, label='train loss', marker='o')
    if val_losses:
        ax.plot(val_steps, val_losses, label='val loss', marker='s')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('Training Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    if train_losses:
        print(f'Last train loss: {train_losses[-1]:.4f}')
    if val_losses:
        print(f'Last val loss:   {val_losses[-1]:.4f}')
        best_epoch = val_steps[val_losses.index(min(val_losses))]
        print(f'Best val loss:   {min(val_losses):.4f}  (epoch {best_epoch})')

## 9 — Check saved checkpoints

In [ ]:
import os, torch
from pathlib import Path

ckpt_dir = Path(OUTPUT_DIR) / 'checkpoints'
checkpoints = sorted(ckpt_dir.glob('*.pt'))

if not checkpoints:
    print('No checkpoints found yet.')
else:
    print(f'Checkpoints in {ckpt_dir}:\n')
    for ckpt in checkpoints:
        size_mb = ckpt.stat().st_size / 1e6
        meta = torch.load(ckpt, map_location='cpu')
        epoch     = meta.get('epoch', '?')
        val_loss  = meta.get('val_loss', float('nan'))
        best_loss = meta.get('best_val_loss', float('nan'))
        print(f'  {ckpt.name:<22}  {size_mb:6.1f} MB  '
              f'epoch={epoch}  val_loss={val_loss:.4f}  best={best_loss:.4f}')

    best_ckpt = ckpt_dir / 'best.pt'
    if best_ckpt.exists():
        print(f'\nBest checkpoint: {best_ckpt}')

## 10 — Resume after Colab disconnect

If the session resets, run cells **1–3** to remount Drive and reinstall deps,  
then run **cell 6** (training) — it auto-detects `last.pt` and resumes.